# 🔫 Gunshot Audio Trimming & Dataset CSV Generator

This notebook creates a robust pipeline that:
1. Scans your `Data` folder for audio files
2. **Trims** gunshot audio (class 1) to isolate only the gunshot event, removing silence
3. Extracts **MFCC features** from all audio files
4. Generates a **randomized CSV** with labels, durations (ms), and features

### Data Mapping
| Folder | Label | Description |
| --- | --- | --- |
| `Data/audio` | 0 | Ambient / environmental sounds |
| `Data/sound` | 0 | Non-gunshot sound effects |
| `Data/gun` | 1 | Gunshot recordings (various firearms) |
| `Data/edge-collected-gunshot-audio` | 1 | Edge-device collected gunshot audio |

In [1]:
# Install dependencies
!pip install -q librosa soundfile tqdm scikit-learn pandas numpy

In [2]:
import os
import warnings
import random
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from pathlib import Path
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")
print("✅ All imports loaded.")

✅ All imports loaded.


## ⚙️ Configuration
Update the paths below to match your data layout.

In [3]:
# ────────────────────── CONFIG ──────────────────────
BASE_DIR = Path(r'c:\Desktop\Data-Cleaner\Data')
OUTPUT_DIR = BASE_DIR / 'Output'
TRIMMED_DIR = OUTPUT_DIR / 'trimmed_gunshots'
CSV_PATH = OUTPUT_DIR / 'dataset_features.csv'

# Class 0 directories (non-gunshot / ambient / noise)
CLASS_0_DIRS = [BASE_DIR / 'audio', BASE_DIR / 'sound']

# Class 1 directories (gunshot)
CLASS_1_DIRS = [BASE_DIR / 'gun', BASE_DIR / 'edge-collected-gunshot-audio']

# MFCC feature extraction settings
SR = 22050          # Sampling rate
N_MFCC = 40         # Number of MFCC coefficients
MAX_LEN = 22050 * 4 # Pad/truncate to 4 seconds for uniformity

# Random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Create output dirs
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRIMMED_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Base directory   : {BASE_DIR}")
print(f"📁 Output directory : {OUTPUT_DIR}")
print(f"📁 Trimmed directory: {TRIMMED_DIR}")

📁 Base directory   : c:\Desktop\Data-Cleaner\Data
📁 Output directory : c:\Desktop\Data-Cleaner\Data\Output
📁 Trimmed directory: c:\Desktop\Data-Cleaner\Data\Output\trimmed_gunshots


## Step 1: Collect Audio Files

In [4]:
def collect_wav_files(directories: list) -> list:
    """Recursively collect all .wav files from a list of directories."""
    files = []
    for d in directories:
        for f in d.rglob('*.wav'):
            files.append(str(f))
    return sorted(files)

class0_files = collect_wav_files(CLASS_0_DIRS)
class1_raw_files = collect_wav_files(CLASS_1_DIRS)

print(f"🎵 Class 0 (non-gunshot) files found : {len(class0_files)}")
print(f"🔫 Class 1 (gunshot) raw files found  : {len(class1_raw_files)}")
print(f"   Total raw files                    : {len(class0_files) + len(class1_raw_files)}")

🎵 Class 0 (non-gunshot) files found : 6230
🔫 Class 1 (gunshot) raw files found  : 4551
   Total raw files                    : 10781


## Step 2: ✂️ Trim Gunshot Audio

Uses `librosa.effects.split` to detect non-silent intervals and keeps only the actual gunshot event. This removes leading/trailing silence and isolates the pure gunshot sound.

Each trimmed file is saved with its **duration in milliseconds** recorded.

In [5]:
def trim_gunshot(filepath: str, output_dir: Path, top_db: int = 20):
    """
    Trim leading/trailing silence from an audio file to isolate the
    gunshot event. Returns (saved_path, duration_ms).
    """
    y, sr = librosa.load(filepath, sr=SR)

    # Use librosa to find non-silent intervals
    intervals = librosa.effects.split(y, top_db=top_db)

    if len(intervals) == 0:
        # File is entirely silent — keep as-is
        trimmed = y
    else:
        # Concatenate all non-silent segments (handles multi-burst shots)
        trimmed = np.concatenate([y[start:end] for start, end in intervals])

    duration_ms = round((len(trimmed) / sr) * 1000, 2)

    # Build output path preserving a unique name
    stem = Path(filepath).stem
    parent_tag = Path(filepath).parent.name
    out_name = f"{parent_tag}__{stem}.wav"
    out_path = output_dir / out_name

    sf.write(str(out_path), trimmed, sr)
    return str(out_path), duration_ms

In [8]:
# Run the trimming
class1_trimmed = []  # (trimmed_path, duration_ms)
errors = []

for f in class1_raw_files:
    try:
        trimmed_path, dur_ms = trim_gunshot(f, TRIMMED_DIR)
        class1_trimmed.append((trimmed_path, dur_ms))
    except Exception as e:
        errors.append((f, str(e)))

print(f"\n✅ Trimmed gunshot clips saved: {len(class1_trimmed)}")
if errors:
    print(f"⚠️  Skipped {len(errors)} files due to errors")
    for path, err in errors[:5]:
        print(f"   • {Path(path).name}: {err}")


✅ Trimmed gunshot clips saved: 4138
⚠️  Skipped 413 files due to errors
   • ._SA_008A_S01.wav: 
   • ._SA_008A_S02.wav: 
   • ._SA_008A_S03.wav: 
   • ._SA_008A_S04.wav: 
   • ._SA_008A_S05.wav: 


In [9]:
# Show trimmed duration statistics
durations = [d for _, d in class1_trimmed]
print("📊 Trimmed Gunshot Duration Stats (milliseconds):")
print(f"   Min     : {min(durations):.2f} ms")
print(f"   Max     : {max(durations):.2f} ms")
print(f"   Mean    : {np.mean(durations):.2f} ms")
print(f"   Median  : {np.median(durations):.2f} ms")
print(f"   Std Dev : {np.std(durations):.2f} ms")

📊 Trimmed Gunshot Duration Stats (milliseconds):
   Min     : 92.88 ms
   Max     : 13075.42 ms
   Mean    : 1635.84 ms
   Median  : 998.46 ms
   Std Dev : 1669.29 ms


## Step 3: 🎶 Extract MFCC Features

Extracts **40 MFCCs** per clip → takes `mean` and `std` across time → **80 features** per sample.

Audio is pad/truncated to 4 seconds for uniformity.

In [ ]:
def extract_mfcc(filepath: str):
    """Extract MFCC features from a .wav file, returning a fixed-length vector."""
    try:
        y, sr = librosa.load(filepath, sr=SR)

        # Pad or truncate to MAX_LEN samples
        if len(y) < MAX_LEN:
            y = np.pad(y, (0, MAX_LEN - len(y)))
        else:
            y = y[:MAX_LEN]

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
        # Summarise each coefficient across time: mean + std → 80 features
        mfcc_mean = np.mean(mfcc, axis=1)
        mfcc_std = np.std(mfcc, axis=1)
        return np.concatenate([mfcc_mean, mfcc_std])
    except Exception as e:
        print(f"  [WARN] Could not process {filepath}: {e}")
        return None

def compute_duration_ms(filepath: str) -> float:
    """Get duration of a wav file in milliseconds."""
    try:
        y, sr = librosa.load(filepath, sr=SR)
        return round((len(y) / sr) * 1000, 2)
    except Exception:
        return 0.0

In [ ]:
n_features = N_MFCC * 2  # mean + std = 80
records = []

# Class 0
print("Processing Class 0 (non-gunshot) ...")
for f in tqdm(class0_files, desc="🎵 Class 0"):
    feat = extract_mfcc(f)
    if feat is not None:
        dur = compute_duration_ms(f)
        records.append({
            'file_path': f,
            'trimmed_duration_ms': dur,
            'label': 0,
            **{f'mfcc_{i}': feat[i] for i in range(n_features)},
        })

# Class 1
print("\nProcessing Class 1 (trimmed gunshots) ...")
for trimmed_path, dur_ms in tqdm(class1_trimmed, desc="🔫 Class 1"):
    feat = extract_mfcc(trimmed_path)
    if feat is not None:
        records.append({
            'file_path': trimmed_path,
            'trimmed_duration_ms': dur_ms,
            'label': 1,
            **{f'mfcc_{i}': feat[i] for i in range(n_features)},
        })

print(f"\n✅ Total feature records extracted: {len(records)}")

## Step 4: 📊 Generate Randomized CSV

Shuffles the entire dataset randomly and saves to CSV with columns:
- `file_path` — path to the audio file
- `trimmed_duration_ms` — duration of the (trimmed) audio in milliseconds
- `label` — 0 (non-gunshot) or 1 (gunshot)
- `mfcc_0` through `mfcc_79` — the 80 MFCC features

In [ ]:
df = pd.DataFrame(records)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
df.to_csv(CSV_PATH, index=False)

n0 = (df['label'] == 0).sum()
n1 = (df['label'] == 1).sum()

print(f"📊 Dataset Summary:")
print(f"   Total samples  : {len(df)}")
print(f"   Class 0 (noise): {n0}")
print(f"   Class 1 (gun)  : {n1}")
print(f"\n💾 CSV saved to: {CSV_PATH}")
df.head()

In [ ]:
# Quick look at label distribution
print("\n🔀 Randomized label distribution (first 20 rows):")
print(df['label'].head(20).tolist())
print(f"\n📈 Label value counts:")
print(df['label'].value_counts())